In [214]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
from sklearn import neighbors
from urllib.request import urlopen
%matplotlib inline

import chart_studio
cs_key = open("./chart_studio_api_key").read()
chart_studio.tools.set_credentials_file(username='Yassminat', 
                                        api_key=cs_key)
import chart_studio.plotly as py

### Loading in dataframe that includes all data as well as addresses and GPS Coords.

In [2]:
df_full = pd.read_csv('data/allegations_precinct_address_included.csv')
df_full.head()

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,...,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,borough,address,long,lat
0,10004,Jonathan,Ruiz,078 PCT,8409,42835,7,2019,5,2020,...,Abuse of Authority,Failure to provide RTKA card,78.0,Report-domestic dispute,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
1,10012,Paula,Smith,078 PCT,4021,37256,5,2017,10,2017,...,Abuse of Authority,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
2,10014,Malachy,Sullivan,078 PCT,4143,33969,11,2015,2,2016,...,Offensive Language,Sexual orientation,78.0,PD suspected C/V of violation/crime - street,Summons - other violation/crime,Substantiated (Formalized Training),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
3,10017,Fazle,Tanim,078 PCT,15187,40070,8,2018,11,2018,...,Discourtesy,Word,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
4,10017,Fazle,Tanim,078 PCT,15187,41927,3,2019,8,2019,...,Abuse of Authority,Refusal to provide shield number,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577


### Loading data layout table as a dataframe for reference 

In [3]:
dlo = pd.read_excel('data/CCRB Data Layout Table.xlsx')
dlo

,field name,description,glossary
0,unique_mos_id,"unique ID of the officer (""member of service"")",NaN
1,first_name,Officer's first name,NaN
2,last_name,Officer's last name,NaN
3,command_now,Officer's command assignment as of July 2020,See Tab 3
4,complaint_id,Unique ID of the complaint,NaN
5,month_received,Month the complaint was received by CCRB,NaN
6,year_received,Year the complaint was received by CCRB,NaN
7,month_closed,Month the complaint investigation was closed b...,NaN
8,year_closed,Year the complaint investigation was closed by...,NaN
9,command_at_incident,Officer's command assignment at the time of th...,NaN


In [4]:
df_full.head()

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,...,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,borough,address,long,lat
0,10004,Jonathan,Ruiz,078 PCT,8409,42835,7,2019,5,2020,...,Abuse of Authority,Failure to provide RTKA card,78.0,Report-domestic dispute,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
1,10012,Paula,Smith,078 PCT,4021,37256,5,2017,10,2017,...,Abuse of Authority,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
2,10014,Malachy,Sullivan,078 PCT,4143,33969,11,2015,2,2016,...,Offensive Language,Sexual orientation,78.0,PD suspected C/V of violation/crime - street,Summons - other violation/crime,Substantiated (Formalized Training),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
3,10017,Fazle,Tanim,078 PCT,15187,40070,8,2018,11,2018,...,Discourtesy,Word,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577
4,10017,Fazle,Tanim,078 PCT,15187,41927,3,2019,8,2019,...,Abuse of Authority,Refusal to provide shield number,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577


In [17]:
# Drop 2020 from dataframe since it's incomeplete data 
df_full = df_full[df_full.year_received<2020]


### Wrangle data to get df grouped year, borough 

In [19]:
## Group by year, borough, and get count
year_borough_grp = df_full.groupby(by=['year_received', 'borough' ],).size()

In [20]:
year_borough_grp.head()

year_received  borough  
1985           Brooklyn      7
1986           Brooklyn     17
               Manhattan     5
1987           Bronx         1
               Brooklyn     16
dtype: int64

In [21]:
## Reset index to get a dataframe from broupby object 
year_borough_grp = year_borough_grp.reset_index()
year_borough_grp.head()

,year_received,borough,0
0,1985,Brooklyn,7
1,1986,Brooklyn,17
2,1986,Manhattan,5
3,1987,Bronx,1
4,1987,Brooklyn,16


In [22]:
## Change name of size(count) column to #_complaints
year_borough_grp.rename(columns = {0:'#_complaints'}, inplace = True) 

In [23]:
year_borough_grp.head()

,year_received,borough,#_complaints
0,1985,Brooklyn,7
1,1986,Brooklyn,17
2,1986,Manhattan,5
3,1987,Bronx,1
4,1987,Brooklyn,16


### Create line plot using plotly
 - using color for boroughs 

In [24]:
fig = px.line(year_borough_grp, x="year_received", y="#_complaints", color="borough",
                 )
fig.show()

### Create line plot for different kinds of complaints 

In [26]:
## Group by year, borough, and get count
year_comp_type_grp = df_full.groupby(by=['year_received', 'fado_type' ],).size()
year_comp_type_grp.head()

year_received  fado_type         
1985           Abuse of Authority    3
               Discourtesy           1
               Force                 3
1986           Abuse of Authority    8
               Discourtesy           4
dtype: int64

In [27]:
# get dataframe from groupby object 
year_comp_type_grp = year_comp_type_grp.reset_index()

In [29]:
year_comp_type_grp.head()

,year_received,fado_type,0
0,1985,Abuse of Authority,3
1,1985,Discourtesy,1
2,1985,Force,3
3,1986,Abuse of Authority,8
4,1986,Discourtesy,4


In [30]:
## Change name of size(count) column to #_complaints
year_comp_type_grp.rename(columns = {0:'#_complaints'}, inplace = True) 

In [31]:
year_comp_type_grp.head()

,year_received,fado_type,#_complaints
0,1985,Abuse of Authority,3
1,1985,Discourtesy,1
2,1985,Force,3
3,1986,Abuse of Authority,8
4,1986,Discourtesy,4


In [37]:
fig = px.line(year_comp_type_grp, x="year_received", y="#_complaints", color="fado_type",)
fig.show()

In [38]:
df_full.unique_mos_id.value_counts()

25861    75
18731    75
18530    73
19489    73
18589    72
         ..
8990      1
11007     1
29183     1
24390     1
9879      1
Name: unique_mos_id, Length: 3994, dtype: int64

In [39]:
mos_id = 25861 
df_25861 = df_full[df_full.unique_mos_id == 25861]

In [41]:
df_25861.head() 

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,...,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,borough,address,long,lat
613,25861,Mathew,Reich,NARCBSI,122,13162,3,2007,4,2008,...,Abuse of Authority,Search (of person),67.0,PD suspected C/V of violation/crime - auto,No arrest made or summons issued,Unsubstantiated,Brooklyn,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665
614,25861,Mathew,Reich,NARCBSI,122,13162,3,2007,4,2008,...,Force,Gun Pointed,67.0,PD suspected C/V of violation/crime - auto,No arrest made or summons issued,Unsubstantiated,Brooklyn,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665
615,25861,Mathew,Reich,NARCBSI,122,13162,3,2007,4,2008,...,Abuse of Authority,Refusal to provide name/shield number,67.0,PD suspected C/V of violation/crime - auto,No arrest made or summons issued,Unsubstantiated,Brooklyn,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665
616,25861,Mathew,Reich,NARCBSI,122,15419,2,2008,2,2009,...,Abuse of Authority,Vehicle search,67.0,PD suspected C/V of violation/crime - auto,Arrest - other violation/crime,Unsubstantiated,Brooklyn,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665
6818,25861,Mathew,Reich,NARCBSI,122,9066,3,2005,10,2005,...,Discourtesy,Word,75.0,PD suspected C/V of violation/crime - street,Summons - disorderly conduct,Exonerated,Brooklyn,"1000 Sutter Avenue, Brooklyn, NY, USA",-73.881581,40.671304


In [66]:
df_25861 = df_25861.groupby(['fado_type']).size()

In [67]:
df_25861.head()

fado_type
Abuse of Authority    11
Discourtesy            8
Force                  8
Offensive Language     1
dtype: int64

In [68]:
df_25861 = df_25861.reset_index()

In [69]:
## Change name of size(count) column to #_complaints
df_25861.rename(columns={0:'#_complaints'}, inplace = True) 

In [70]:
df_25861.head()

,fado_type,#_complaints
0,Abuse of Authority,11
1,Discourtesy,8
2,Force,8
3,Offensive Language,1


In [74]:
fig = px.bar(df_25861, x="fado_type", y="#_complaints", color="fado_type")
fig.show()

In [75]:
fig = px.scatter(df_25861, x="fado_type", y="#_complaints", color="fado_type", size='#_complaints')
fig.show()

## Create and fill unsubstantiated/exonerated column 

In [78]:
board_dis_vals = df_full.board_disposition.value_counts()
board_dis_vals

Unsubstantiated                             15428
Exonerated                                   9597
Substantiated (Charges)                      3790
Substantiated (Formalized Training)          1033
Substantiated (Command Discipline A)          962
Substantiated (Command Discipline)            851
Substantiated (Command Discipline B)          784
Substantiated (Command Lvl Instructions)      452
Substantiated (Instructions)                  247
Substantiated (No Recommendations)            161
Substantiated (MOS Unidentified)                1
Name: board_disposition, dtype: int64

In [81]:
board_dis_vals = pd.DataFrame(board_dis_vals)

In [84]:
board_dis_vals = board_dis_vals.reset_index()

In [85]:
board_dis_vals

,index,board_disposition
0,Unsubstantiated,15428
1,Exonerated,9597
2,Substantiated (Charges),3790
3,Substantiated (Formalized Training),1033
4,Substantiated (Command Discipline A),962
5,Substantiated (Command Discipline),851
6,Substantiated (Command Discipline B),784
7,Substantiated (Command Lvl Instructions),452
8,Substantiated (Instructions),247
9,Substantiated (No Recommendations),161


In [87]:
board_dis_vals.rename(columns={'index': 'board_disposition', 'board_disposition': 'total'}, inplace=True)

In [88]:
board_dis_vals

,board_disposition,total
0,Unsubstantiated,15428
1,Exonerated,9597
2,Substantiated (Charges),3790
3,Substantiated (Formalized Training),1033
4,Substantiated (Command Discipline A),962
5,Substantiated (Command Discipline),851
6,Substantiated (Command Discipline B),784
7,Substantiated (Command Lvl Instructions),452
8,Substantiated (Instructions),247
9,Substantiated (No Recommendations),161


In [90]:
fig = px.bar(board_dis_vals, x="board_disposition", y="total", color="board_disposition")
fig.show()

In [91]:
board_dis_vals

,board_disposition,total
0,Unsubstantiated,15428
1,Exonerated,9597
2,Substantiated (Charges),3790
3,Substantiated (Formalized Training),1033
4,Substantiated (Command Discipline A),962
5,Substantiated (Command Discipline),851
6,Substantiated (Command Discipline B),784
7,Substantiated (Command Lvl Instructions),452
8,Substantiated (Instructions),247
9,Substantiated (No Recommendations),161


In [93]:
unsub_exonerated = ['Unsubstantiated', 'Exonerated']
condition = df_full.board_disposition.isin(unsub_exonerated)
df_full['unsubstantiated/exonerated'] = condition

In [94]:
df_full.head()

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,...,allegation,precinct,contact_reason,outcome_description,board_disposition,borough,address,long,lat,unsubstantiated/exonerated
0,10004,Jonathan,Ruiz,078 PCT,8409,42835,7,2019,5,2020,...,Failure to provide RTKA card,78.0,Report-domestic dispute,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577,False
1,10012,Paula,Smith,078 PCT,4021,37256,5,2017,10,2017,...,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577,False
2,10014,Malachy,Sullivan,078 PCT,4143,33969,11,2015,2,2016,...,Sexual orientation,78.0,PD suspected C/V of violation/crime - street,Summons - other violation/crime,Substantiated (Formalized Training),Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577,False
3,10017,Fazle,Tanim,078 PCT,15187,40070,8,2018,11,2018,...,Word,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577,True
4,10017,Fazle,Tanim,078 PCT,15187,41927,3,2019,8,2019,...,Refusal to provide shield number,78.0,Moving violation,Moving violation summons issued,Unsubstantiated,Brooklyn,"65 6th Avenue, Brooklyn, NY, USA",-73.974198,40.680577,True


In [140]:
df_25861 = df_full[df_full.unique_mos_id == 25861]

In [141]:
df_25861 = df_25861.fado_type.value_counts()
df_25861 = pd.DataFrame(df_25861)

In [143]:
df_25861 = df_25861.reset_index()
# df_25861 = df_25861.T
df_25861

,index,fado_type
0,Abuse of Authority,50
1,Force,12
2,Discourtesy,12
3,Offensive Language,1


In [102]:
df_25861.rename(columns={'index': 'Category', 'fado_type': 'count'}, inplace=True)
df_25861 = df_25861.T

In [112]:
df_25861.reset_index()

,index,Abuse of Authority,Force,Discourtesy,Offensive Language
0,fado_type,50,12,12,1


In [147]:
df_tmp = df_full[df_full.unique_mos_id == 25861]
df_25861['total'] = len(df_tmp)

In [119]:
df_25861

,Abuse of Authority,Force,Discourtesy,Offensive Language,total
fado_type,50,12,12,1,75


In [120]:
df_tmp.head()

,unique_mos_id,first_name,last_name,command_now,shield_no,complaint_id,month_received,year_received,month_closed,year_closed,...,allegation,precinct,contact_reason,outcome_description,board_disposition,borough,address,long,lat,unsubstantiated/exonerated
613,25861,Mathew,Reich,NARCBSI,122,13162,3,2007,4,2008,...,Search (of person),67.0,PD suspected C/V of violation/crime - auto,No arrest made or summons issued,Unsubstantiated,Brooklyn,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,True
614,25861,Mathew,Reich,NARCBSI,122,13162,3,2007,4,2008,...,Gun Pointed,67.0,PD suspected C/V of violation/crime - auto,No arrest made or summons issued,Unsubstantiated,Brooklyn,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,True
615,25861,Mathew,Reich,NARCBSI,122,13162,3,2007,4,2008,...,Refusal to provide name/shield number,67.0,PD suspected C/V of violation/crime - auto,No arrest made or summons issued,Unsubstantiated,Brooklyn,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,True
616,25861,Mathew,Reich,NARCBSI,122,15419,2,2008,2,2009,...,Vehicle search,67.0,PD suspected C/V of violation/crime - auto,Arrest - other violation/crime,Unsubstantiated,Brooklyn,"2820 Snyder Avenue, Brooklyn, NY, USA",-73.950264,40.648665,True
6818,25861,Mathew,Reich,NARCBSI,122,9066,3,2005,10,2005,...,Word,75.0,PD suspected C/V of violation/crime - street,Summons - disorderly conduct,Exonerated,Brooklyn,"1000 Sutter Avenue, Brooklyn, NY, USA",-73.881581,40.671304,True


In [123]:
len(df_tmp[df_tmp['unsubstantiated/exonerated'] == True])

68

In [148]:
df_25861['# Unsubstantiated/Exonerated'] = len(df_tmp[df_tmp['unsubstantiated/exonerated'] == True])

In [149]:
df_25861

,index,fado_type,# Unsubstantiated/Exonerated,total
0,Abuse of Authority,50,68,75
1,Force,12,68,75
2,Discourtesy,12,68,75
3,Offensive Language,1,68,75


In [150]:
df_25861['# Substantiated'] = df_25861['total'] - df_25861['# Unsubstantiated/Exonerated']

In [139]:
df_25861 = df_25861.T.reset_index()
df_25861

,index,0,1,2,3,4,5,6,7,8
0,index,index,0,1,2,3,4,5,6,7
1,0,index,index,0,1,2,3,4,5,6
2,1,0,index,Abuse of Authority,Force,Discourtesy,Offensive Language,total,# Unsubstantiated/Exonerated,# Substantiated
3,2,1,fado_type,50,12,12,1,75,68,7


In [170]:
df_tmp2 = df_full[df_full.unique_mos_id == 25861]

In [171]:
df_tmp2.shape

(75, 32)

In [172]:
df_tmp2 = df_tmp2.groupby(['fado_type', 'unsubstantiated/exonerated']).size()
df_tmp2 = df_tmp2.reset_index()

In [173]:
df_tmp2.rename(columns={0:'count'}, inplace=True)
df_tmp2['count'].sum()

75

In [174]:
df_tmp2['count'].sum()

75

In [175]:
df_tmp2.head()

,fado_type,unsubstantiated/exonerated,count
0,Abuse of Authority,False,6
1,Abuse of Authority,True,44
2,Discourtesy,True,12
3,Force,False,1
4,Force,True,11


In [194]:
labels = {'fado_type': 'Complaint Type', 
         'unsubstantiated/exonerated': 'Board Found Unsubstantiated',}
fig = px.bar(df_tmp2, x="fado_type", y="count", 
             color="unsubstantiated/exonerated",
            labels=labels)
fig.show()
py.plot(fig)

'https://plotly.com/~Yassminat/19/'

In [196]:
labels = {'fado_type': 'Complaint Type', 
         'unsubstantiated/exonerated': 'Board Found Unsubstantiated',}
fig = px.bar(df_tmp2, x="unsubstantiated/exonerated", y="count", 
             color="fado_type",
            labels=labels)
fig.show()
py.plot(fig)

'https://plotly.com/~Yassminat/23/'

In [205]:
#Create labels to be displayed 
labels = {'fado_type': 'Complaint Type', 
         'unsubstantiated/exonerated': 'Board Found Unsubstantiated',}

fig = px.bar(df_tmp2, x='count', y="unsubstantiated/exonerated",
             color='fado_type', barmode='group', labels = labels, orientation='h' )
fig.show()
# py.plot(fig)

In [206]:
labels = {'fado_type': 'Complaint Type', 
         'unsubstantiated/exonerated': 'Board Found Unsubstantiated',}
fig = px.bar(df_tmp2, x="count", y="unsubstantiated/exonerated", 
             color="fado_type",
            labels=labels)
fig.show()
py.plot(fig)

'https://plotly.com/~Yassminat/27/'

In [201]:
labels = {'fado_type': 'Complaint Type', 
         'unsubstantiated/exonerated': 'Board Found Unsubstantiated',}

fig = px.scatter(df_tmp2, x='fado_type', y="count",
             color='unsubstantiated/exonerated', size='count', labels = labels )
fig.show()
# py.plot(fig)

In [207]:
df = px.data.tips()
fig = px.treemap(df, path=['day', 'time', 'sex'], values='total_bill')
fig.show()

In [208]:
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [209]:
df_tmp2.head()

,fado_type,unsubstantiated/exonerated,count
0,Abuse of Authority,False,6
1,Abuse of Authority,True,44
2,Discourtesy,True,12
3,Force,False,1
4,Force,True,11


In [212]:
fig = px.treemap(df_tmp2, path=['unsubstantiated/exonerated', 'fado_type'], values='count',
                color='fado_type')
fig.show()
py.plot(fig)

'https://plotly.com/~Yassminat/33/'